# Binary AutoML pipeline

Полный path-based lifecycle: typed config, train, model layouts, predict, evaluate, persistence и optional Osiris execution.

In [ ]:
from pathlib import Path

from avatar.automl import EnvironmentConfig, BinaryTask, BinaryTaskConfig

train_path = Path('/shared/data/train')
valid_path = Path('/shared/data/valid')
test_path = Path('/shared/data/test')
output_dir = Path('outputs/binary')

In [ ]:
config = BinaryTaskConfig(
    env_type='local',
    backend='boosting',
    engine='catboost',
    device='gpu',
    target_column='target',
    client_id_column='epk_id',
    group_column='group',
    date_column='report_month',
    categorical_columns=['cat_feature_1', 'cat_feature_2'],
    numerical_columns=['num_feature_1', 'num_feature_2'],
    hidden_state_columns=['seq_hidden_state'],
    model_layout='global_and_per_group',
    hyperopt=True,
    optimization_metric='roc_auc',
    n_trials=50,
    output_dir=output_dir,
)

task = BinaryTask(config)

In [ ]:
training = task.train(train_path, valid_path)
training

## Model layouts

Artifact с обеими ветками можно использовать целиком либо явно выбрать одну поддержанную ветку.

In [ ]:
predictions_by_layout = {
    layout: task.predict(test_path, model_layout=layout)
    for layout in ('global', 'per_group', 'global_and_per_group')
}
prediction = predictions_by_layout['global_and_per_group']
prediction.scores.head()

In [ ]:
evaluation = task.evaluate(
    test_path,
    prediction,
    metrics=['roc_auc', 'precision@5', 'recall@5', 'precision@10', 'recall@10', 'precision@20', 'recall@20', 'precision@25', 'recall@25', 'precision@50', 'recall@50'],
)
artifact_path = task.save()

restored = BinaryTask.load(artifact_path)
restored_prediction = restored.predict(test_path)
evaluation.metrics_raw

## Osiris

Для standard profile опустите custom resources. Custom pool требует явные `num_nodes` и `num_gpus`.

In [ ]:
restored.predict(
    test_path,
    env_type='osiris',
    environment=EnvironmentConfig(pool='b2c', num_nodes=1, num_gpus=4),
)
restored.status(wait=True)
remote_prediction = restored.load_prediction(test_path)